In [1]:
import os
import gc
import torch
from pathlib import Path
import json
import pandas as pd
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion

In [2]:
data_yaml = "D:\\LabsMIET\\ML-Detection-object\\dataset\\data.yaml"
project_name = "miet_5fold"
test_images_dir = "D:\\LabsMIET\\ML-Detection-object\\test_images\\test_images"
solution_csv = "D:\\LabsMIET\\ML-Detection-object\\sample_sub.csv"
output_Submission_csv = "submission_wbf.csv"

In [3]:
def clear_memory_on_epoch_end(trainer):
    gc.collect()             
    torch.cuda.empty_cache()

In [4]:
def yolo_to_voc(x_center, y_center, w, h):
    """Конвертирует YOLO [xc, yc, w, h] в WBF [xmin, ymin, xmax, ymax]"""
    return [
        max(0.0, x_center - w / 2.0),
        max(0.0, y_center - h / 2.0),
        min(1.0, x_center + w / 2.0),
        min(1.0, y_center + h / 2.0)
    ]

def voc_to_yolo(xmin, ymin, xmax, ymax):
    """Конвертирует WBF [xmin, ymin, xmax, ymax] обратно в YOLO [xc, yc, w, h]"""
    w = xmax - xmin
    h = ymax - ymin
    xc = xmin + w / 2.0
    yc = ymin + h / 2.0
    return [xc, yc, w, h]

In [4]:
import pandas as pd
from pathlib import Path
from ensemble_boxes import weighted_boxes_fusion

def yolo_to_voc(xc, yc, w, h):
    """Перевод из YOLO (center_x, center_y, width, height) в VOC (x_min, y_min, x_max, y_max) с клиппингом [0, 1]"""
    x_min = xc - (w / 2)
    y_min = yc - (h / 2)
    x_max = xc + (w / 2)
    y_max = yc + (h / 2)
    
    # Обязательно обрезаем координаты для WBF, иначе будет ошибка ValueError
    return [
        max(0.0, min(1.0, x_min)),
        max(0.0, min(1.0, y_min)),
        max(0.0, min(1.0, x_max)),
        max(0.0, min(1.0, y_max))
    ]

def voc_to_yolo(x_min, y_min, x_max, y_max):
    """Перевод из VOC обратно в YOLO"""
    xc = (x_min + x_max) / 2
    yc = (y_min + y_max) / 2
    w = x_max - x_min
    h = y_max - y_min
    return xc, yc, w, h

def build_wbf_submission(
    solution_csv: str,
    preds_dirs_list: list,
    output_csv: str = "submission_wbf.csv",
    image_col: str = "image_name",
    boxes_col: str = "boxes",  # Колонка, куда запишем результат
    row_id_col: str = "id",
    iou_thr: float = 0.5,        
    skip_box_thr: float = 0.01, 
    keep_only_class: int | None = None
):
    sol_path = Path(solution_csv)
    if not sol_path.exists():
        raise FileNotFoundError(f"Файл solution_csv не найден: {sol_path}")

    sol = pd.read_csv(sol_path)
    if image_col not in sol.columns:
        raise ValueError(f"В {solution_csv} нет колонки '{image_col}'")

    rows = []
    # Веса для моделей (одинаковые для всех, если не переданы иные)
    weights = [1.0] * len(preds_dirs_list)

    # Итерируемся по строкам оригинального датафрейма
    for idx, row in sol.iterrows():
        image_name = str(row[image_col])
        row_id = row[row_id_col] if row_id_col in sol.columns else idx
        
        stem = Path(image_name).stem
        
        boxes_list = []
        scores_list = []
        labels_list = []

        # Сбор предсказаний из всех папок
        for pdir_str in preds_dirs_list:
            pred_file = Path(pdir_str) / f"{stem}.txt"
            
            model_boxes = []
            model_scores = []
            model_labels = []

            if pred_file.exists():
                content = pred_file.read_text(encoding="utf-8", errors="replace").strip()
                if content:
                    for ln in content.splitlines():
                        parts = ln.split()
                        if len(parts) < 5: 
                            continue
                        
                        try:
                            cls = int(float(parts[0]))
                            if keep_only_class is not None and cls != keep_only_class:
                                continue
                            
                            xc, yc, w, h = map(float, parts[1:5])
                            # В YOLO предсказаниях 6-й элемент обычно score. Если его нет, ставим 1.0
                            sc = float(parts[5]) if len(parts) >= 6 else 1.0
                            
                            bbox = yolo_to_voc(xc, yc, w, h)
                            
                            model_boxes.append(bbox)
                            model_scores.append(sc)
                            model_labels.append(cls)
                        except ValueError:
                            continue
            
            boxes_list.append(model_boxes)
            scores_list.append(model_scores)
            labels_list.append(model_labels)

        final_yolo_boxes_str = []
        total_boxes_found = sum(len(b) for b in boxes_list)
        
        # Применяем WBF, если хоть одна модель нашла хоть один бокс
        if total_boxes_found > 0:
            merged_boxes, merged_scores, merged_labels = weighted_boxes_fusion(
                boxes_list, scores_list, labels_list, 
                weights=weights, iou_thr=iou_thr, skip_box_thr=skip_box_thr
            )
            
            for bbox, score, label in zip(merged_boxes, merged_scores, merged_labels):
                xc, yc, w, h = voc_to_yolo(*bbox)
                final_score = max(0.0, min(1.0, float(score)))
                
                # Формируем строку: класс уверенность x_center y_center ширина высота
                # (Адаптируйте этот формат под то, как требует ваше соревнование)
                final_yolo_boxes_str.append(f"{int(label)} {final_score:.6f} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}")

        # Склеиваем все боксы в одну строку через пробел
        prediction_string = " ".join(final_yolo_boxes_str)

        rows.append({
            row_id_col: row_id,
            image_col: image_name,
            boxes_col: prediction_string
        })

    # Создаем финальный датафрейм и сохраняем
    result_df = pd.DataFrame(rows)
    result_df.to_csv(output_csv, index=False)
    print(f"Файл успешно сохранен: {output_csv}")

In [10]:
from pathlib import Path
import json
import pandas as pd
from ensemble_boxes import weighted_boxes_fusion

def yolo_to_voc(xc, yc, w, h):
    """Конвертация YOLO [xc, yc, w, h] в VOC [x1, y1, x2, y2]"""
    x1 = xc - w / 2
    y1 = yc - h / 2
    x2 = xc + w / 2
    y2 = yc + h / 2
    # Клиппинг для предотвращения ошибок в ensemble_boxes
    return [max(0.0, min(1.0, x1)), max(0.0, min(1.0, y1)), 
            max(0.0, min(1.0, x2)), max(0.0, min(1.0, y2))]

def voc_to_yolo(x1, y1, x2, y2):
    """Конвертация VOC [x1, y1, x2, y2] в YOLO [xc, yc, w, h]"""
    xc = (x1 + x2) / 2
    yc = (y1 + y2) / 2
    w = x2 - x1
    h = y2 - y1
    return [xc, yc, w, h]

def build_wbf_submission(
    solution_csv: str,
    preds_dirs: list,  # список путей к папкам с предсказаниями
    output_csv: str = "submission_wbf.csv",
    image_col: str = "image_name",
    boxes_col: str = "boxes",
    row_id_col: str = "id",
    iou_thr: float = 0.5,
    skip_box_thr: float = 0.0001,
    keep_only_class: int | None = None
) -> None:
    sol = pd.read_csv(solution_csv)
    image_names = sol[image_col].astype(str).tolist()
    weights = [1.0] * len(preds_dirs)  # Можно настроить веса для каждой папки

    rows = []
    for idx, image_name in enumerate(image_names):
        stem = Path(image_name).stem
        
        # Списки для WBF
        all_boxes = []
        all_scores = []
        all_labels = []

        for pdir in preds_dirs:
            pred_file = Path(pdir) / f"{stem}.txt"
            model_boxes, model_scores, model_labels = [], [], []
            
            if pred_file.exists():
                for ln in pred_file.read_text().splitlines():
                    parts = ln.split()
                    if len(parts) < 5: continue
                    
                    cls, xc, yc, w, h = map(float, parts[:5])
                    sc = float(parts[5]) if len(parts) >= 6 else 1.0
                    
                    if keep_only_class is not None and int(cls) != keep_only_class:
                        continue
                    
                    model_boxes.append(yolo_to_voc(xc, yc, w, h))
                    model_scores.append(sc)
                    model_labels.append(int(cls))
            
            all_boxes.append(model_boxes)
            all_scores.append(model_scores)
            all_labels.append(model_labels)

        # Выполнение WBF
        final_boxes_json = []
        if any(len(b) > 0 for b in all_boxes):
            merged_boxes, merged_scores, merged_labels = weighted_boxes_fusion(
                all_boxes, all_scores, all_labels, 
                weights=weights, iou_thr=iou_thr, skip_box_thr=skip_box_thr
            )
            
            for i in range(len(merged_boxes)):
                xc, yc, w, h = voc_to_yolo(*merged_boxes[i])
                final_boxes_json.append([round(xc, 6), round(yc, 6), round(w, 6), round(h, 6), round(merged_scores[i], 6)])

        rows.append({
            row_id_col: idx,
            image_col: image_name,
            boxes_col: json.dumps(final_boxes_json, separators=(",", ":"))
        })

    pd.DataFrame(rows).to_csv(output_csv, index=False)
    print(f"WBF submission saved to {output_csv}")

# Использование:
# build_wbf_submission("sample_sub.csv", ["path/to/fold1", "path/to/fold2", "path/to/fold3"])

In [ ]:
num_folds = 3
n_epochs = 50
trained_models_weights = []

base_model_name = "yolo26s.pt" 

for fold in range(1, num_folds + 1):
    print(f"\n Фолд {fold}/{num_folds} ")
        
    model = YOLO(base_model_name)
    model.add_callback("on_train_epoch_end", clear_memory_on_epoch_end)

    run_name = f"fold_{fold}"

    results = model.train(
        data=data_yaml,
        epochs=n_epochs,            
        imgsz=640,
        device='cuda',
        name=run_name,
        project=project_name,
        workers=0,             
        batch=-1,              
        patience=10,           
        cache=False,            
        seed=42 + 5*fold,        
        
        optimizer="AdamW",
        cos_lr=True,

        pretrained=True,

        weight_decay=0.01,
        
        # Аугментации для обучения 
        mosaic=1.0,            # Склейка 4 картинок в одну
        close_mosaic=10,      # Вероятность отключить склейку
        mixup=0.1,             # Смешивание изображений
        degrees=15.0,          # Поворот
        hsv_s=0.5,             # Изменение насыщенности
        hsv_v=0.4,             # Изменение яркости 
        fliplr=0.5             # Отзеркаливание по горизонтали
    )
        
    best_weight_path = Path(results.save_dir) / "weights" / "best.pt"
    trained_models_weights.append(str(best_weight_path))

    del model
    gc.collect()                 
    torch.cuda.empty_cache()


 Фолд 1/3 
Ultralytics 8.4.21  Python-3.13.7 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3050, 8191MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\LabsMIET\ML-Detection-object\dataset\data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolo26m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=fold_14, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, 

In [7]:
prediction_dirs = []

for fold_idx, weight_path in enumerate(trained_models_weights, start=1):
    print(f"Предсказание моделью {fold_idx}: {weight_path}")
    model = YOLO(weight_path)
        
    run_name = f"predict_fold_{fold_idx}"
        
    res = model.predict(
        source=test_images_dir,
        save_txt=True,
        save_conf=True,
        stream=True,
        augment=False,      
        conf=0.15,          
        iou=0.6,
        project='miet_5fold',
        name=run_name
    )
        
    for _ in res:
        pass
        
    pred_labels_dir = Path(project_name) / run_name / "labels"
    prediction_dirs.append(str(pred_labels_dir))

    del model
    gc.collect()
    torch.cuda.empty_cache()


Предсказание моделью 1: D:\LabsMIET\ML-Detection-object\runs\detect\miet_5fold\fold_14\weights\best.pt

WARNING Model does not support 'augment=True', reverting to single-scale prediction.
image 1/4454 D:\LabsMIET\ML-Detection-object\test_images\test_images\1.1_00-00-00-000.jpg: 384x640 1 customer, 62.2ms
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
image 2/4454 D:\LabsMIET\ML-Detection-object\test_images\test_images\1.1_00-00-12-500.jpg: 384x640 2 customers, 15.5ms
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
image 3/4454 D:\LabsMIET\ML-Detection-object\test_images\test_images\1.1_00-00-25-000.jpg: 384x640 2 customers, 15.4ms
WARNING Model does not support 'augment=True', reverting to single-scale prediction.
image 4/4454 D:\LabsMIET\ML-Detection-object\test_images\test_images\1.1_00-00-37-500.jpg: 384x640 2 customers, 17.6ms
WARNING Model does not support 'augment=True', reverting to single-scale predicti

In [11]:
prediction_dirs = ['runs\\detect\\miet_5fold\\predict_fold_1\\labels', 'runs\\detect\\miet_5fold\\predict_fold_2\\labels', 'runs\\detect\\miet_5fold\\predict_fold_3\\labels']
build_wbf_submission(
    solution_csv=solution_csv,
    preds_dirs_list=prediction_dirs,
    output_csv=output_Submission_csv,
    iou_thr=0.5,           
    skip_box_thr=0.01,     
    keep_only_class=1      
)

TypeError: build_wbf_submission() got an unexpected keyword argument 'preds_dirs_list'

In [12]:
build_wbf_submission("sample_sub.csv", prediction_dirs, "submission_wbf_test.csv", iou_thr=0.5, skip_box_thr=0.01, keep_only_class=1)

WBF submission saved to submission_wbf_test.csv
